In [1]:
# environment setup: install tf-keras shim, disable tf backend, install rag stack
import os, subprocess, sys

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

def _pip(*pkgs):
    try:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", *pkgs],
            capture_output=True, text=True, timeout=600,
        )
        if r.returncode != 0:
            print(f"[pip] EXIT {r.returncode} for {pkgs}")
            for ln in (r.stderr or "").strip().splitlines()[-5:]:
                print(" ", ln)
    except Exception as e:
        print(f"[pip] {pkgs} failed: {e}")

_pip("tf-keras")
_pip("transformers>=4.40,<5", "accelerate>=0.27", "sentencepiece")
_pip("sentence-transformers")
_pip("rank-bm25")
_pip("wandb")

print("env ready")

env ready


In [2]:
# imports, seeds, paths, hyperparameters
import os, re, gc, json, time, random, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PROBE_SIZE            = 200
TRAIN_CORPUS_ROWS     = 1800
TOP_K_EXTERNAL        = 3
TOP_K_INTERNAL        = 3
CHUNK_MAX_WORDS       = 120
EMBEDDER_NAME         = "sentence-transformers/all-MiniLM-L6-v2"
LLM_NAME              = "Qwen/Qwen2.5-1.5B-Instruct"
WIKIPEDIA_TOP_N_TOPICS = 100

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# wrapper-stripping normalizer and train load
_PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Determine the correct option:",
    "Identify the correct statement:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]
_SUFFIXES = [
    "among the listed options.",
    "based on the given context.",
    "from the following choices.",
    "carefully.",
    "from the choices given.",
    "from the options provided.",
]
_PREFIX_RE = re.compile(r"^\s*(?:" + "|".join(re.escape(p) for p in _PREFIXES) + r")\s*")
_SUFFIX_RE = re.compile(r"\s*(?:" + "|".join(re.escape(s) for s in _SUFFIXES) + r")\s*$")
_WS_RE = re.compile(r"\s+")

def normalize_stem(text):
    if not isinstance(text, str):
        return text
    s = text
    while True:
        before = s
        s = _PREFIX_RE.sub("", s)
        s = _SUFFIX_RE.sub("", s)
        if s == before:
            break
    return _WS_RE.sub(" ", s).strip()

train = pd.read_csv(DATA_DIR / "train.csv")
train["core_q"] = train["prompt"].apply(normalize_stem)

print(f"train rows: {len(train)}")
print(f"unique core_q: {train['core_q'].nunique()}")
train.head(2)

train rows: 2000
unique core_q: 252


,id,prompt,A,B,C,D,E,answer,core_q
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,What is Martin Heidegger's view on the relatio...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,What is accelerator-based light-ion fusion?


In [4]:
# extract topic seeds from prompts and fetch wikipedia summaries
import requests

_NOUN_RE = re.compile(r"\b[A-Z][a-z]{3,}(?:\s+[A-Z][a-z]+)*\b")
_STOP_NOUNS = {"Which", "What", "Who", "The", "This", "That", "How", "Where", "When",
               "Pick", "Select", "Determine", "Identify", "Choose"}

def extract_topics(prompts, top_n=100):
    counts = {}
    for p in prompts:
        for m in _NOUN_RE.findall(str(p)):
            if m in _STOP_NOUNS:
                continue
            counts[m] = counts.get(m, 0) + 1
    return [t for t, _ in sorted(counts.items(), key=lambda x: -x[1])[:top_n]]

def fetch_wikipedia_passages(topics, cache_path=None):
    if cache_path and Path(cache_path).exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)
    passages = []
    for t in topics:
        slug = t.replace(" ", "_")
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}"
        try:
            r = requests.get(url, timeout=10, headers={"User-Agent": "mcq-rag/1.0"})
            if r.status_code == 200:
                d = r.json()
                extract = (d.get("extract") or "").strip()
                if len(extract) > 100:
                    passages.append({"id": slug, "title": d.get("title", t), "text": extract})
        except Exception:
            continue
    if cache_path:
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(passages, f, ensure_ascii=False, indent=2)
    return passages

seed_topics = extract_topics(train["prompt"].tolist(), top_n=WIKIPEDIA_TOP_N_TOPICS)
print(f"seed topics: {len(seed_topics)}")
print("top 20:", seed_topics[:20])

wiki_cache = OUT_DIR / "rag_wikipedia_corpus.json"
external_passages = fetch_wikipedia_passages(seed_topics, cache_path=wiki_cache)
print(f"external passages fetched: {len(external_passages)}")

seed topics: 90
top 20: ['Earth', 'Einstein', 'Newton', 'Modified Newtonian Dynamics', 'Lorentz', 'Navier', 'Stokes', 'Isaac Newton', 'Gravity Probe', 'Pierre', 'Fermat', 'Doppler', 'Minkowski', 'Maxwell', 'Demon', 'Liouville', 'Relativity', 'Kutta', 'Peierls', 'Optical Signal']
external passages fetched: 11


In [5]:
# sentence-boundary chunker and external chunk list
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")

def split_sentences(text):
    text = (text or "").strip()
    if not text:
        return []
    return [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]

def chunk_passage(text, max_words=120, overlap_sentences=1):
    sents = split_sentences(text)
    if not sents:
        return []
    chunks, cur, cur_words = [], [], 0
    for s in sents:
        w = len(s.split())
        if cur and cur_words + w > max_words:
            chunks.append(" ".join(cur))
            if overlap_sentences and len(cur) >= overlap_sentences:
                cur = cur[-overlap_sentences:]
                cur_words = sum(len(x.split()) for x in cur)
            else:
                cur, cur_words = [], 0
        cur.append(s)
        cur_words += w
    if cur:
        chunks.append(" ".join(cur))
    return chunks

external_chunks = []
for p in external_passages:
    for i, ch in enumerate(chunk_passage(p["text"], max_words=CHUNK_MAX_WORDS, overlap_sentences=1)):
        external_chunks.append({"id": f"wiki::{p['id']}::{i}", "text": ch})

print(f"external chunks: {len(external_chunks)}")
if external_chunks:
    print(f"mean words: {np.mean([len(c['text'].split()) for c in external_chunks]):.1f}")

external chunks: 12
mean words: 63.7


In [6]:
# build internal corpus from first 1800 train rows as Q/A passages
train_for_corpus = train.head(TRAIN_CORPUS_ROWS).copy()

internal_passages = []
for _, r in train_for_corpus.iterrows():
    correct_text = str(r[r["answer"]])
    passage_text = f"Q: {r['prompt']}\nA: {correct_text}"
    internal_passages.append({
        "id": f"train::{r['id']}",
        "text": passage_text,
    })

print(f"internal passages: {len(internal_passages)}")
print()
print("example:")
print(internal_passages[0]["text"][:400])

internal passages: 1800

example:
Q: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
A: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or enga


In [7]:
# hybrid bm25 + minilm dense index, build both external and internal
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

_TOKEN_RE = re.compile(r"[a-z0-9]+")

def bm25_tokenize(text):
    return _TOKEN_RE.findall((text or "").lower())

class HybridIndex:
    def __init__(self, passages, embedder):
        self.passages = passages
        self.tokenized = [bm25_tokenize(p["text"]) for p in passages]
        self.bm25 = BM25Okapi(self.tokenized)
        self.embedder = embedder
        self.dense = embedder.encode(
            [p["text"] for p in passages],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=64,
        )

    def _norm(self, hits):
        if not hits:
            return {}
        vals = np.array([s for _, s in hits], dtype=np.float32)
        lo, hi = vals.min(), vals.max()
        rng = (hi - lo) if hi > lo else 1.0
        return {i: (s - lo) / rng for i, s in hits}

    def search(self, query, top_k=5, alpha=0.5):
        bm25_scores = self.bm25.get_scores(bm25_tokenize(query))
        bm25_idx = np.argsort(bm25_scores)[::-1][:top_k * 3]
        bm25_hits = [(int(i), float(bm25_scores[i])) for i in bm25_idx]
        q = self.embedder.encode(
            [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
        )[0]
        dense_scores = self.dense @ q
        dense_idx = np.argsort(dense_scores)[::-1][:top_k * 3]
        dense_hits = [(int(i), float(dense_scores[i])) for i in dense_idx]
        n_bm = self._norm(bm25_hits)
        n_dn = self._norm(dense_hits)
        keys = set(n_bm) | set(n_dn)
        fused = {k: (1 - alpha) * n_bm.get(k, 0.0) + alpha * n_dn.get(k, 0.0) for k in keys}
        ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return [(k, float(v)) for k, v in ranked]

    def get(self, idx):
        return self.passages[idx]

print("loading embedder...")
embedder = SentenceTransformer(EMBEDDER_NAME, device=DEVICE)
print("embedder ready")

print()
print("building external index...")
external_index = HybridIndex(external_chunks, embedder) if external_chunks else None
print(f"external ready: {len(external_chunks)} chunks")

print("building internal index...")
internal_index = HybridIndex(internal_passages, embedder)
print(f"internal ready: {len(internal_passages)} passages")

loading embedder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedder ready

building external index...
external ready: 12 chunks
building internal index...
internal ready: 1800 passages


In [8]:
# load qwen 2.5 1.5b instruct and resolve letter token ids
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"loading {LLM_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map=DEVICE,
)
model.eval()

LETTER_IDS = {}
for L in "ABCDE":
    ids = tokenizer.encode(L, add_special_tokens=False)
    LETTER_IDS[L] = ids[0]

print(f"params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"letter token IDs: {LETTER_IDS}")

loading Qwen/Qwen2.5-1.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

params: 1.54B
letter token IDs: {'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36}


In [9]:
# prompt template and constrained letter scoring
SYSTEM_PROMPT = (
    "You are an expert answering a multiple-choice question. "
    "You will be shown some background context and a similar-question reference, "
    "then a question and five options labelled A through E. "
    "Choose the single best option. Respond with exactly one letter: A, B, C, D, or E."
)

def build_message(external_ctx, internal_ctx, prompt, options):
    ext_block = "\n\n".join(f"- {c}" for c in external_ctx) if external_ctx else "(none)"
    int_block = "\n\n".join(internal_ctx) if internal_ctx else "(none)"
    opt_block = "\n".join(f"{L}. {options[L]}" for L in "ABCDE")
    return (
        f"BACKGROUND CONTEXT:\n{ext_block}\n\n"
        f"SIMILAR QUESTIONS FROM TRAINING:\n{int_block}\n\n"
        f"QUESTION: {prompt}\n\n"
        f"OPTIONS:\n{opt_block}\n\n"
        f"Answer with a single letter:"
    )

@torch.inference_mode()
def score_letters(user_msg):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=3072).to(DEVICE)
    logits = model(**inputs).logits[0, -1, :]
    letter_logits = torch.tensor([logits[LETTER_IDS[L]].item() for L in "ABCDE"])
    probs = torch.softmax(letter_logits, dim=0).cpu().numpy()
    return {L: float(probs[i]) for i, L in enumerate("ABCDE")}

_test = score_letters(build_message(
    external_ctx=["Isaac Newton formulated three laws of motion in the 17th century."],
    internal_ctx=["Q: Who formulated the laws of motion?\nA: Isaac Newton"],
    prompt="Who formulated the classical laws of motion?",
    options={"A": "Albert Einstein", "B": "Isaac Newton", "C": "Galileo Galilei",
             "D": "James Maxwell", "E": "Niels Bohr"},
))
print("smoke test:", {k: round(v, 3) for k, v in _test.items()})
print("predicted:", max(_test, key=_test.get))

smoke test: {'A': 0.0, 'B': 1.0, 'C': 0.0, 'D': 0.0, 'E': 0.0}
predicted: B


In [10]:
# wandb authentication via kaggle secrets, fallback to offline
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("wandb_api_key")
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key, verify=True)
    print("wandb: authenticated")
except Exception as e:
    print(f"wandb: no secret found ({e}); running offline")
    os.environ["WANDB_MODE"] = "offline"

wandb: no secret found (Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 126501038 and label wandb_api_key.'], 'error': {'code': 5}, 'wasSuccessful': False}.); running offline


In [11]:
# score the 200-row probe with both external and internal retrieval
probe = train.tail(PROBE_SIZE).reset_index(drop=True)
print(f"probe rows: {len(probe)}")

wandb.init(
    project="23f2001849-t22026",
    name="rag_llm_qwen1p5b_two_corpus",
    tags=["rag", "llm", "qwen2.5-1.5b", "two-corpus"],
    config={
        "llm": LLM_NAME,
        "embedder": EMBEDDER_NAME,
        "top_k_external": TOP_K_EXTERNAL,
        "top_k_internal": TOP_K_INTERNAL,
        "chunk_max_words": CHUNK_MAX_WORDS,
        "probe_size": PROBE_SIZE,
        "external_chunks": len(external_chunks),
        "internal_passages": len(internal_passages),
    },
)

def top3(letter_probs):
    order = sorted(letter_probs.items(), key=lambda x: -x[1])
    return " ".join(L for L, _ in order[:3])

preds = []
t0 = time.time()
for i, row in probe.iterrows():
    options = {L: str(row[L]) for L in "ABCDE"}
    query = row["prompt"] + " " + " ".join(options.values())

    ext_hits = external_index.search(query, top_k=TOP_K_EXTERNAL) if external_index else []
    ext_ctx = [external_index.get(idx)["text"] for idx, _ in ext_hits]

    int_hits = internal_index.search(query, top_k=TOP_K_INTERNAL)
    int_ctx = [internal_index.get(idx)["text"] for idx, _ in int_hits]

    msg = build_message(ext_ctx, int_ctx, row["prompt"], options)
    probs = score_letters(msg)
    preds.append(top3(probs))

    if (i + 1) % 20 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(probe) - i - 1)
        print(f"{i+1}/{len(probe)}  elapsed={elapsed:.0f}s  eta={eta:.0f}s")

probe["pred_top3"] = preds
print()
print(f"total: {time.time() - t0:.1f}s")

probe rows: 200


wandb: Tracking run with wandb version 0.26.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20260709_183545-5frybofm


20/200  elapsed=6s  eta=52s
40/200  elapsed=11s  eta=45s
60/200  elapsed=16s  eta=38s
80/200  elapsed=21s  eta=32s
100/200  elapsed=27s  eta=27s
120/200  elapsed=33s  eta=22s
140/200  elapsed=38s  eta=16s
160/200  elapsed=43s  eta=11s
180/200  elapsed=50s  eta=6s
200/200  elapsed=56s  eta=0s

total: 55.7s


In [12]:
# map@3 evaluation and per-tier breakdown against 4-tier lookup
def map_at_3(y_true, y_pred_top3):
    total = 0.0
    for true, pred in zip(y_true, y_pred_top3):
        for rank, L in enumerate(pred.split()[:3], start=1):
            if L == true:
                total += 1.0 / rank
                break
    return total / len(y_true)

rag_score = map_at_3(probe["answer"].tolist(), probe["pred_top3"].tolist())
prior_score = map_at_3(probe["answer"].tolist(), ["B C A"] * len(probe))
print(f"RAG+LLM MAP@3:      {rag_score:.4f}")
print(f"Label-prior MAP@3:  {prior_score:.4f}")
print(f"Lift over prior:    {rag_score - prior_score:+.4f}")

wandb.log({
    "map_at_3": rag_score,
    "label_prior_map_at_3": prior_score,
    "lift_over_prior": rag_score - prior_score,
})

def option_set(row):
    return tuple(sorted(str(row[c]) for c in "ABCDE"))

def build_tables(df):
    T1, T2 = {}, {}
    core_correct = defaultdict(set)
    tc, tw = Counter(), Counter()
    for _, r in df.iterrows():
        os_ = option_set(r)
        correct = str(r[r["answer"]])
        T1[(r["core_q"], os_)] = correct
        T2[os_] = correct
        core_correct[r["core_q"]].add(correct)
        for L in "ABCDE":
            (tc if L == r["answer"] else tw)[str(r[L])] += 1
    T3 = {t for t in tc if tw.get(t, 0) == 0}
    return {"T1": T1, "T2": T2, "T3": T3, "T4": dict(core_correct)}

def resolve_tier(row, tables):
    cq = row["core_q"]; os_ = option_set(row)
    cand = {L: str(row[L]) for L in "ABCDE"}
    t = tables["T1"].get((cq, os_))
    if t is not None and sum(1 for v in cand.values() if v == t) == 1:
        return "T1"
    t = tables["T2"].get(os_)
    if t is not None and sum(1 for v in cand.values() if v == t) == 1:
        return "T2"
    if sum(1 for v in cand.values() if v in tables["T3"]) == 1:
        return "T3"
    known = tables["T4"].get(cq, set())
    if sum(1 for v in cand.values() if v in known) == 1:
        return "T4"
    return "MISS"

tables = build_tables(train.head(TRAIN_CORPUS_ROWS))
probe["tier"] = probe.apply(lambda r: resolve_tier(r, tables), axis=1)

print()
print("probe counts by tier:")
print(probe["tier"].value_counts())

print()
print("MAP@3 by tier:")
for tier in ["T1", "T2", "T3", "T4", "MISS"]:
    sub = probe[probe["tier"] == tier]
    if len(sub) > 0:
        s = map_at_3(sub["answer"].tolist(), sub["pred_top3"].tolist())
        print(f"{tier}: {s:.4f}  (n={len(sub)})")
        wandb.log({f"map_at_3_{tier}": s, f"n_{tier}": int(len(sub))})

RAG+LLM MAP@3:      0.9392
Label-prior MAP@3:  0.4308
Lift over prior:    +0.5083

probe counts by tier:
tier
T1      152
T3       41
MISS      4
T2        3
Name: count, dtype: int64

MAP@3 by tier:
T1: 0.9550  (n=152)
T2: 1.0000  (n=3)
T3: 0.8943  (n=41)
MISS: 0.7500  (n=4)


In [13]:
# persist predictions, finish wandb, free gpu memory
preds_path = OUT_DIR / "rag_llm_probe_predictions.csv"
probe[["id", "prompt", "answer", "pred_top3", "tier"]].to_csv(preds_path, index=False)
print(f"saved: {preds_path}")

wandb.finish()

del model, embedder
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print()
print("done.")

wandb: 
wandb: Run history:
wandb: label_prior_map_at_3 ▁
wandb:      lift_over_prior ▁
wandb:             map_at_3 ▁
wandb:        map_at_3_MISS ▁
wandb:          map_at_3_T1 ▁
wandb:          map_at_3_T2 ▁
wandb:          map_at_3_T3 ▁
wandb:               n_MISS ▁
wandb:                 n_T1 ▁
wandb:                 n_T2 ▁
wandb:                   +1 ...
wandb: 
wandb: Run summary:
wandb: label_prior_map_at_3 0.43083
wandb:      lift_over_prior 0.50833
wandb:             map_at_3 0.93917
wandb:        map_at_3_MISS 0.75
wandb:          map_at_3_T1 0.95504
wandb:          map_at_3_T2 1
wandb:          map_at_3_T3 0.89431
wandb:               n_MISS 4
wandb:                 n_T1 152
wandb:                 n_T2 3
wandb:                   +1 ...
wandb: 
wandb: You can sync this run to the cloud by running:
wandb: wandb sync /kaggle/working/wandb/offline-run-20260709_183545-5frybofm
wandb: Find logs at: ./wandb/offline-run-20260709_183545-5frybofm/logs


saved: /kaggle/working/rag_llm_probe_predictions.csv

done.
